# Chapter 8: Hamiltonian Perturbations

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 8, printed pp. 257-294; PDF pp. 272-309. Sections: 8.1-8.6.

    ## Chapter Goal

    Trivial bundles, locally Hamiltonian fibrations, pseudoholomorphic sections, fiber spheres, section pseudocycles, and section counts.

    The guiding question is:

    > How does a Hamiltonian perturbation turn curve counting into a theory of sections over Riemann surfaces?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| Hamiltonian fibration | bundle with symplectic fibers and connection | holonomy check |
| perturbed CR equation | ordinary defect plus Hamiltonian vector term | residual norm |
| section | graph over the base | one point in each fiber |
| vertical bubble | sphere inside a fiber | fiber energy |
| section count | zero-dimensional section moduli | constraint balance |


## Standalone Reading Guide

    Hamiltonian perturbations broaden the curve equation without abandoning symplectic control. Instead of maps into a fixed target only, the chapter considers sections of fibrations whose fibers are symplectic manifolds. A connection and a Hamiltonian term modify the Cauchy-Riemann equation so that section counts can encode monodromy and Hamiltonian dynamics.

The geometric picture is a graph over a Riemann surface. Each point of the base chooses a point in the fiber. The perturbation tells the section how to twist as it moves around the base. Vertical bubbles can still appear, so compactness and pseudocycle ideas return in a fiberwise form. Counting sections is a bridge to applications such as Hamiltonian periodic orbits and representations built from loops of Hamiltonian diffeomorphisms.

The experiment uses a Hamiltonian on a two-torus. Its vector field is divergence-free with respect to the area form, and the quiver plot shows the local direction in which the perturbation pushes a section. This is a finite-dimensional shadow of the bundle equation: the Hamiltonian term is not noise but structured symplectic motion.

    ## Topics In This Notebook

    - trivial and locally Hamiltonian fibrations
- Hamiltonian terms in the section equation
- pseudoholomorphic sections and vertical spheres
- section pseudocycles and fixed-domain counts
- connections to periodic orbits and later Seidel-type constructions

    ## Visualization Storyboard

    - A torus Hamiltonian vector-field plot displays the perturbation term that enters the section equation.
- A dependency graph connects fibrations, sections, bubbles, and counts.
- A ledger checks symplectic divergence and action variation for the model Hamiltonian.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-08'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['Hamiltonian fibration', 'perturbed CR equation', 'section', 'vertical bubble', 'section count']
CONCEPT_EDGES = [('Hamiltonian fibration', 'perturbed CR equation'), ('perturbed CR equation', 'section'), ('section', 'vertical bubble'), ('vertical bubble', 'section count')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=35, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Hamiltonian Perturbations')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '257-294',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Hamiltonian Perturbations. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
n = 25
x = np.linspace(0, 1, n)
y = np.linspace(0, 1, n)
X, Y = np.meshgrid(x, y)
Hx = -2 * np.pi * np.sin(2 * np.pi * X)
Hy = np.pi * np.cos(2 * np.pi * Y)
U = Hy
V = -Hx
dy_u, dx_u = np.gradient(U, y, x)
dy_v, dx_v = np.gradient(V, y, x)
divergence = dx_u + dy_v

fig, ax = plt.subplots(figsize=(6.2, 5.4))
speed = np.sqrt(U**2 + V**2)
ax.quiver(X, Y, U, V, speed, cmap="plasma", scale=120)
ax.set_aspect("equal")
ax.set_title("Hamiltonian vector field on a torus chart")
ax.set_xlabel("base/fiber coordinate x")
ax.set_ylabel("base/fiber coordinate y")
fig_path = save_matplotlib(fig, UNIT, "figures", "hamiltonian-vector-field-section-perturbation.png")
plt.close(fig)

check = {
    "max_abs_discrete_divergence": float(np.abs(divergence[2:-2, 2:-2]).max()),
    "mean_speed": float(speed.mean()),
    "periodic_boundary_match_x": float(np.abs(U[:, 0] - U[:, -1]).max()),
    "passed": bool(np.abs(divergence[2:-2, 2:-2]).max() < 1e-10 and np.abs(U[:, 0] - U[:, -1]).max() < 1e-10),
}
check_path = save_json(check, UNIT, "checks", "hamiltonian-vector-field-checks.json")
display_artifact(fig_path, width=620)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'Hamiltonian fibration', 'computational_object': 'bundle with symplectic fibers and connection', 'check': 'holonomy check'}, {'item': 'perturbed CR equation', 'computational_object': 'ordinary defect plus Hamiltonian vector term', 'check': 'residual norm'}, {'item': 'section', 'computational_object': 'graph over the base', 'check': 'one point in each fiber'}, {'item': 'vertical bubble', 'computational_object': 'sphere inside a fiber', 'check': 'fiber energy'}, {'item': 'section count', 'computational_object': 'zero-dimensional section moduli', 'check': 'constraint balance'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Replace the Hamiltonian by a nonperiodic polynomial on the square. The divergence check may still pass locally, but the torus interpretation and boundary matching will break.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Hamiltonian perturbations encode symplectic monodromy inside a curve equation.
- Section counts reuse compactness and transversality in a fibered setting.
- Vertical bubbles are the fiberwise compactness issue to track.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'hamiltonian-vector-field-section-perturbation.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'hamiltonian-vector-field-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'hamiltonian-vector-field-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
